# ETAPE 1 : IMPORTATIONS ET INITIALISATION DE L'ENVIRONNEMENT

In [ ]:
# --- IMPORTATION DES LIBRAIRIES ---

import numpy as np
import pandas as pd
import random
from faker import Faker
import os

fake = Faker("fr_FR")

np.random.seed(42)
random.seed(42)

# --- CONFIGURATION GLOBALE ---
CONFIG = {
    # Temps et calendrier
    "calendar": {
        "start_date": "2020-01-01",
        "end_date": "2025-12-31",
        "weekend_days": [4, 5],
        "closed_days": [0, 6],
        "operatings_hours": [8, 23],
        "holiday_months": [8],
        "winter_months": [1, 2, 11, 12],
        "exceptional_closure_periods":{
            "covid": {
                "years": [2020, 2021],
                "reduction_rate": 0.6,
                "closure_periods": [
                    {"start": "2020-03-17", "end": "2020-05-11"},
                    {"start": "2020-10-30", "end": "2020-12-15"},
                    {"start": "2021-04-03", "end": "2021-05-03"}
                ]
            }
        }
    },
    
    # Activité et flux
    "activity": {
        "n_customers": 8000,
        "activity_coefficients": {0: 0.0, # Lundi
                                  1: 0.8, # Mardi
                                  2: 0.9, # Mercredi
                                  3: 0.9, # Jeudi
                                  4: 1.2, # Vendredi
                                  5: 1.2, # Samedi
                                  6: 0.0  # Dimanche
                                  },
        "services": {
            "SRV001": {"name": "Déjeuner", "start": 12, "end": 14, "rotation": 2, "occupancy_rate": [0.50, 0.85]},
            "SRV002": {"name": "Cocktails", "start": 14, "end": 19, "rotation": 5, "occupancy_rate": [0.30, 0.50]},
            "SRV003": {"name": "Dîner", "start": 19, "end": 23, "rotation": 2, "occupancy_rate": [0.65, 0.95]}
        },
    },

    # Infrastructure
    "infrastructure": {
        "tables": {
            "Salle": {"2": 20, "4": 15},
            "Terrasse": {"2": 15, "4": 5},
            "VIP": {"10": 1}
            },
        
        "employees": {
            "n_teams": 2,
            "schedules": {
                "Salle": {1: {"start": 10, "end": 17}, 2: {"start": 16, "end": 23}},
                "Cuisine": {1: {"start": 8, "end": 16}, 2: {"start": 15, "end": 23}},
            },

            "roles": {
                "Chef de cuisine": {"Department": "Cuisine", "salary": 3500, "hours": 151.67, "count": 1},
                "Second de cuisine": {"Department": "Cuisine", "salary": 2600, "hours": 151.67, "count": 1},
                "Chef de partie": {"Department": "Cuisine", "salary": 2100, "hours": 151.67, "count": 2},
                "Commis de cuisine": {"Department": "Cuisine", "salary": 1750, "hours": 151.67, "count": 3},
                "Plongeur": {"Department": "Cuisine", "salary": 1700, "hours": 151.67, "count": 2},
        
                "Directeur de salle": {"Department": "Salle", "salary": 3200, "hours": 151.67, "count": 1},
                "Maître d'hôtel": {"Department": "Salle", "salary": 2500, "hours": 151.67, "count": 1},
                "Chef de rang": {"Department": "Salle", "salary": 2100, "hours": 151.67, "count": 3},
                "Serveur": {"Department": "Salle", "salary": 1900, "hours": 151.67, "count": 5},
                "Commis de salle": {"Department": "Salle", "salary": 1800, "hours": 151.67, "count": 3},
                "Runner": {"Department": "Salle", "salary": 1750, "hours": 151.67, "count": 2}
            },
        
            "role_weights": {
                # Les rôles de salle
                "Serveur": 0.50,
                "Chef de rang": 0.30,
                "Runner": 0.10,
                "Commis de salle": 0.05,
            
                # Encadrement
                "Maître d'hôtel": 0.025,      
                "Directeur de salle": 0.025,
            },
        
            "restrictions": {
                "Cuisine": ["SRV002"]
            }
            },
    },

    # Commerce et clientèle
    "commercial": {
        "promotions": {
            "PRM001": {"name": "Aucune promotion", "discount": 0.00, "prob": 0.80},
            "PRM002": {"name": "Menu Déjeuner", "discount": 0.10, "prob": 0.10},
            "PRM003": {"name": "Happy Hour", "discount": 0.15, "prob": 0.10}
        },
        
        "customer_segments": {
            "types": ["Local", "Touriste", "Professionnel"],
            "probs": [0.6, 0.25, 0.15]
        },
        
        "behavior": {
            "pax": {"sizes": [1, 2, 3, 4, 6, 8], "probs": [0.05, 0.55, 0.15, 0.20, 0.03, 0.02]},
            "food_qty": [1, 2],
             "customer_profiles": {"price_sensitivity_range": [0.3, 1.0], "visit_frequency_lambda": 2},
             "probabilities": {"wine": 0.55, "starter": 0.5, "dessert": 0.50, "drink": 0.40, "tip": 0.35}
        }   
    },

    # Gestion financière
    "financials": {
        "restock_freq": 250,
        "wastage_freq": 800,
        "margin_limit": 0.7,
        "payment_methods": {"Espèces": 0.05, "Visa": 0.45, "Mastercard": 0.35, "Apple Pay": 0.15},
        "tips": {"min_rate": 0.03, "max_rate": 0.12},
        "stock_management": {"loss_probability": 0.12,
                             "loss_rate_range": [0.02, 0.40],
                             "volatility_factor": 0.40},
    },

    # Système
     "system": {
         "output_folder": "Data/Raw"
     }
}

# Création du dossier d'export automatiquement
if not os.path.exists(CONFIG["system"]["output_folder"]):
    os.makedirs(CONFIG["system"]["output_folder"])

# ETAPE 2 : DEFINITION DU CATALOGUE PRODUIT

In [ ]:
MENU_DATA = {
    # --- ENTRÉES & PARTAGE ---
    "Planche Charcuterie": {
        "cat": "Entrée", "type": "Charcuterie", "price_bin": "Standard", "base_price": 16.0,
        "recipe": {"Jambon": {"qty": 50, "unit": "g"}, "Saucisson": {"qty": 50, "unit": "g"}, "Pain": {"qty": 50, "unit": "g"}}
    },
    "Burrata à la Truffe": {
        "cat": "Entrée", "type": "Fromage", "price_bin": "Premium", "base_price": 19.5,
        "recipe": {"Burrata": {"qty": 125, "unit": "g"}, "Truffe": {"qty": 2, "unit": "g"}, "Tomates": {"qty": 50, "unit": "g"}, "Huile Olive": {"qty": 10, "unit": "ml"}}
    },
    "Ceviche de Dorade": {
        "cat": "Entrée", "type": "Poisson", "price_bin": "Premium", "base_price": 18.0,
        "recipe": {"Dorade": {"qty": 150, "unit": "g"}, "Citron vert": {"qty": 15, "unit": "g"}, "Oignon": {"qty": 20, "unit": "g"}, "Coriandre": {"qty": 2, "unit": "g"}}
    },
    "Escargots Bourgogne": {
        "cat": "Entrée", "type": "Spécialité", "price_bin": "Standard", "base_price": 15.0,
        "recipe": {"Escargots": {"qty": 6, "unit": "u"}, "Beurre": {"qty": 10, "unit": "g"}, "Ail": {"qty": 2, "unit": "g"}, "Persil": {"qty": 2, "unit": "g"}}
    },
    
    # --- PIZZAS ARTISANALES ---
    "Pizza Margherita": {
        "cat": "Plat", "type": "Pizza", "price_bin": "Budget", "base_price": 12.0,
        "recipe": {"Farine": {"qty": 200, "unit": "g"}, "Tomates": {"qty": 50, "unit": "g"}, "Mozza": {"qty": 80, "unit": "g"}, "Basilic": {"qty": 2, "unit": "g"}}
    },
    "Pizza Diavola": {
        "cat": "Plat", "type": "Pizza", "price_bin": "Budget", "base_price": 14.5,
        "recipe": {"Farine": {"qty": 200, "unit": "g"}, "Tomates": {"qty": 50, "unit": "g"}, "Mozza": {"qty": 80, "unit": "g"}, "Salami": {"qty": 40, "unit": "g"}}
    },
    "Pizza Truffe & Champ": {
        "cat": "Plat", "type": "Pizza", "price_bin": "Premium", "base_price": 21.0,
        "recipe": {"Farine": {"qty": 200, "unit": "g"}, "Mozza": {"qty": 80, "unit": "g"}, "Truffe": {"qty": 2, "unit": "g"}, "Champignons": {"qty": 40, "unit": "g"}}
    },
    "Pizza 4 Fromages": {
        "cat": "Plat", "type": "Pizza", "price_bin": "Standard", "base_price": 16.0,
        "recipe": {"Farine": {"qty": 200, "unit": "g"}, "Emmental": {"qty": 40, "unit": "g"}, "Mozza": {"qty": 60, "unit": "g"}, "Gorgonzola": {"qty": 30, "unit": "g"}, "Chèvre": {"qty": 30, "unit": "g"}}
    },

    # --- PLATS PRINCIPAUX ---
    "Steak Frites Poivre": {
        "cat": "Plat", "type": "Viande", "price_bin": "Standard", "base_price": 22.0,
        "recipe": {"Bœuf": {"qty": 150, "unit": "g"}, "Pommes de terre": {"qty": 150, "unit": "g"}, "Poivre": {"qty": 2, "unit": "g"}, "Crème": {"qty": 20, "unit": "ml"}}
    },
    "Côte de Bœuf (1kg)": {
        "cat": "Plat", "type": "Viande", "price_bin": "Luxe", "base_price": 65.0,
        "recipe": {"Bœuf": {"qty": 1000, "unit": "g"}, "Pommes de terre": {"qty": 150, "unit": "g"}, "Fleur de sel": {"qty": 2, "unit": "g"}, "Herbes": {"qty": 2, "unit": "g"}}
    },
    "Filet de Bar Rôti": {
        "cat": "Plat", "type": "Poisson", "price_bin": "Premium", "base_price": 26.0,
        "recipe": {"Bar": {"qty": 150, "unit": "g"}, "Citron": {"qty": 15, "unit": "g"}, "Fenouil": {"qty": 150, "unit": "g"}}
    },
    "Linguine aux Palourdes": {
        "cat": "Plat", "type": "Pâtes", "price_bin": "Standard", "base_price": 19.0,
        "recipe": {"Linguine": {"qty": 100, "unit": "g"}, "Palourdes": {"qty": 500, "unit": "g"}, "Ail": {"qty": 2, "unit": "g"}, "Vin blanc": {"qty": 30, "unit": "ml"}}
    },
    "Risotto Asperges": {
        "cat": "Plat", "type": "Spécialité", "price_bin": "Standard", "base_price": 17.5,
        "recipe": {"Riz": {"qty": 80, "unit": "g"}, "Asperges": {"qty": 80, "unit": "g"}, "Parmesan": {"qty": 30, "unit": "g"}}
    },
    "Curry Vegan Coco": {
        "cat": "Plat", "type": "Végétarien", "price_bin": "Standard", "base_price": 16.0,
        "recipe": {"Tofu": {"qty": 150, "unit": "g"}, "Lait coco": {"qty": 100, "unit": "ml"}, "Curry": {"qty": 5, "unit": "g"}, "Riz": {"qty": 80, "unit": "g"}}
    },
    "Confit de Canard": {
        "cat": "Plat", "type": "Viande", "price_bin": "Standard", "base_price": 20.0,
        "recipe": {"Canard": {"qty": 150, "unit": "g"}, "Graisse": {"qty": 5, "unit": "g"}, "Pommes de terre": {"qty": 150, "unit": "g"}}
    },

    # --- DESSERTS ---
    "Fondant Chocolat": {
        "cat": "Dessert", "type": "Pâtisserie", "price_bin": "Budget", "base_price": 8.0,
        "recipe": {"Chocolat": {"qty": 50, "unit": "g"}, "Beurre": {"qty": 10, "unit": "g"}, "Œufs": {"qty": 1, "unit": "u"}, "Farine": {"qty": 15, "unit": "g"}, "Sucre": {"qty": 20, "unit": "g"}}
    },
    "Pavlova Fruits Rouges": {
        "cat": "Dessert", "type": "Pâtisserie", "price_bin": "Standard", "base_price": 11.5,
        "recipe": {"Meringue": {"qty": 1, "unit": "u"}, "Crème": {"qty": 20, "unit": "ml"}, "Fruits": {"qty": 100, "unit": "g"}}
    },
    "Tarte Citron Meringuée": {
        "cat": "Dessert", "type": "Pâtisserie", "price_bin": "Budget", "base_price": 9.0,
        "recipe": {"Pâte": {"qty": 80, "unit": "g"}, "Citron": {"qty": 15, "unit": "g"}, "Sucre": {"qty": 10, "unit": "g"}}
    },
    "Assiette de Fromages": {
        "cat": "Dessert", "type": "Fromage", "price_bin": "Standard", "base_price": 12.0,
        "recipe": {"Fromage": {"qty": 100, "unit": "g"}, "Noix": {"qty": 10, "unit": "g"}}
    },

    # --- VINS (L'accord correspond au 'type' du plat) ---
    "Bordeaux Grand Cru (Verre)":     {"cat": "Boisson", "type": "Vin", "price_bin": "Luxe", "base_price": 10.0, "accord": "Viande", "recipe": {"Bordeaux Grand Cru":{"qty": 125, "unit": "ml"}}},
    "Vin Rouge Maison (Verre)":       {"cat": "Boisson", "type": "Vin", "price_bin": "Budget", "base_price": 5.0, "accord": "Viande", "recipe": {"Vin Rouge Maison":{"qty": 125, "unit": "ml"}}},
    "Chablis Premier Cru (Verre)":    {"cat": "Boisson", "type": "Vin", "price_bin": "Premium", "base_price": 7.0, "accord": "Poisson", "recipe": {"Chablis Premier Cru":{"qty": 125, "unit": "ml"}}},
    "Sancerre Blanc (Verre)":         {"cat": "Boisson", "type": "Vin", "price_bin": "Premium", "base_price": 7.0, "accord": "Poisson", "recipe": {"Sancerre Blanc":{"qty": 125, "unit": "ml"}}},
    "Champagne Cuvée (Verre)":        {"cat": "Boisson", "type": "Vin", "price_bin": "Luxe", "base_price": 16.0, "accord": "Poisson", "recipe": {"Champagne Cuvée":{"qty": 125, "unit": "ml"}}},

    "Bordeaux Grand Cru":     {"cat": "Boisson", "type": "Vin", "price_bin": "Luxe", "base_price": 55.0, "accord": "Viande", "recipe": {"Bordeaux Grand Cru":{"qty": 750, "unit": "ml"}}},
    "Vin Rouge Maison":       {"cat": "Boisson", "type": "Vin", "price_bin": "Budget", "base_price": 15.0, "accord": "Viande", "recipe": {"Vin Rouge Maison":{"qty": 750, "unit": "ml"}}},
    "Chablis Premier Cru":    {"cat": "Boisson", "type": "Vin", "price_bin": "Premium", "base_price": 42.0, "accord": "Poisson", "recipe": {"Chablis Premier Cru":{"qty": 750, "unit": "ml"}}},
    "Sancerre Blanc":         {"cat": "Boisson", "type": "Vin", "price_bin": "Premium", "base_price": 38.0, "accord": "Poisson", "recipe": {"Sancerre Blanc":{"qty": 750, "unit": "ml"}}},
    "Champagne Cuvée":        {"cat": "Boisson", "type": "Vin", "price_bin": "Luxe", "base_price": 95.0, "accord": "Poisson", "recipe": {"Champagne Cuvée":{"qty": 750, "unit": "ml"}}},

    # --- BOISSONS & DIGESTIFS ---
    "Espresso Double":        {"cat": "Boisson", "type": "Café", "price_bin": "Budget", "base_price": 4.0, "recipe": {"Café": {"qty": 7, "unit": "g"}}},
    "Cocktail Spritz":        {"cat": "Boisson", "type": "Cocktail", "price_bin": "Standard", "base_price": 10.0, "recipe": {"Prosecco": {"qty": 100, "unit": "ml"}, "Apérol": {"qty": 30, "unit": "ml"}}},
    "Jus Pomme Ginger":       {"cat": "Boisson", "type": "Jus", "price_bin": "Budget", "base_price": 6.0, "recipe": {"Pomme": {"qty": 100, "unit": "g"}, "Gingembre": {"qty": 5, "unit": "g"}}},
    "Digestif Cognac":        {"cat": "Boisson", "type": "Alcool", "price_bin": "Premium", "base_price": 12.0, "recipe": {"Cognac": {"qty": 40, "unit": "ml"}}},
    "Thé Infusion":           {"cat": "Boisson", "type": "Thé", "price_bin": "Budget", "base_price": 5.0, "recipe": {"Thé": {"qty": 2, "unit": "g"}}}
}

INGS_COSTS = {
    "Ail": 0.008, "Apérol": 0.015, "Asperges": 0.012, "Bar": 0.028, 
    "Basilic": 0.020, "Beurre": 0.009, "Bordeaux Grand Cru": 0.03, "Burrata": 0.060, "Bœuf": 0.025,
    "Café": 0.050, "Canard": 0.018, "Chablis Premier Cru": 0.02, "Champagne Cuvée": 0.05, "Champignons": 0.015, "Chocolat": 0.015,
    "Chèvre": 0.018, "Citron": 0.005, "Citron vert": 0.008, "Cognac": 0.040, 
    "Coriandre": 0.025, "Crème": 0.006, "Curry": 0.030, "Dorade": 0.022, 
    "Emmental": 0.012, "Escargots": 0.700, "Farine": 0.001, "Fenouil": 0.004, 
    "Fleur de sel": 0.010, "Fromage": 0.020, "Fruits": 0.015, "Gingembre": 0.015, 
    "Gorgonzola": 0.018, "Graisse": 0.005, "Herbes": 0.030, "Huile Olive": 0.018, 
    "Jambon": 0.022, "Lait coco": 0.008, "Linguine": 0.004, "Meringue": 0.500, 
    "Mozza": 0.015, "Noix": 0.025, "Oignon": 0.002, "Pain": 0.003, 
    "Palourdes": 0.018, "Parmesan": 0.022, "Persil": 0.015, "Poivre": 0.040, 
    "Pomme": 0.003, "Pommes de terre": 0.002, "Prosecco": 0.012, "Pâte": 0.006, 
    "Riz": 0.003, "Salami": 0.020, "Sancerre Blanc": 0.019, "Saucisson": 0.025, "Sucre": 0.002,
    "Thé": 0.050, "Tofu": 0.012, "Tomates": 0.004, "Truffe": 2.500, 
    "Vin blanc": 0.015, "Vin Rouge Maison": 0.007, "Œufs": 0.300
}

# Vérification des ingrédients du menu dans la table ingrédients
ings_menu = {ing for v in MENU_DATA.values() for ing in v.get("recipe", {})}
ings_missing = [ing for ing in ings_menu if ing not in INGS_COSTS]

if len(ings_missing) > 0:
    print(f"⚠️ Attention, les ingrédients suivants n'ont pas de coût défini : {ings_missing}")
else:
    print("✅ Tous les ingrédients sont couverts par le dictionnaire INGS_COSTS.")

# ETAPE 3 : CONSTRUCTION DES TABLES REFERENTIELS

In [ ]:
# ======================================
# 0. INITIALISATION D'UN DICTIONNAIRE DF
# ======================================

df_dict={}

# ============
# 1. CUSTOMERS
# ============

def build_customers(CONFIG):
    
    n_customers=CONFIG["activity"]["n_customers"]
    segments=CONFIG["commercial"]["customer_segments"]
    profiles=CONFIG["commercial"]["behavior"]["customer_profiles"]
    
    data_customers=[]
    
    for i in range(1, n_customers+1):
        data_customers.append({
            "customer_id": i,
            "customer_code": f"CUS{i:03d}",
            "customer_name": fake.name(),
            "customer_type": np.random.choice(segments["types"], p=segments["probs"]),
            "customer_price_sensitivity": round(np.random.uniform(profiles["price_sensitivity_range"][0], 
                                                         profiles["price_sensitivity_range"][1]), 2),
            "customer_visit_frequency": np.random.poisson(2)
        })

    return pd.DataFrame(data_customers)

df_dict["customers"] = build_customers(CONFIG)

# ============
# 2. EMPLOYEES
# ============

def build_employees(CONFIG):
    n_teams = CONFIG["infrastructure"]["employees"]["n_teams"]
    employee_roles = CONFIG["infrastructure"]["employees"]["roles"]

    data_employees = []
    employee_id=1

    for team_id in range(1, n_teams+1):
        for role, info in employee_roles.items():
            count = info.get("count", 1)

            for _ in range(count):
                data_employees.append({
                    "employee_id": employee_id,
                    "employee_code": f"EMP{employee_id:03d}",
                    "employee_name": fake.name(),
                    "employee_role": role,
                    "employee_team": team_id,
                    "employee_department": info["Department"],
                    "employee_fixed_monthly_salary": info["salary"],
                    "employee_monthly_hours_contract": info["hours"],
                    "employee_hourly_rate": round(info["salary"] / info["hours"], 2)
                })
                employee_id += 1

    return pd.DataFrame(data_employees) 

df_dict["employees"]=build_employees(CONFIG)

# ==============
# 3. INGREDIENTS
# ==============

def build_ingredients(menu_data, ings_costs):
    data_ingredients = []
    added_ingredients = set()
    ing_id = 1
    
    for details in menu_data.values():
        for name, info in details.get("recipe", {}).items():
            if name not in added_ingredients:
                data_ingredients.append({
                    "ingredient_id": ing_id,
                    "ingredient_code": f"ING{ing_id:03d}",
                    "ingredient_name": name,
                    "ingredient_unit": info["unit"],
                    "ingredient_unit_cost": ings_costs.get(name, 0.0)
                })
                added_ingredients.add(name)
                ing_id += 1
                
    return pd.DataFrame(data_ingredients)

df_dict["ingredients"]=build_ingredients(MENU_DATA, INGS_COSTS)

# ===========
# 4. PRODUCTS
# ===========

def build_products(menu_data):
    data_products = []
    product_id = 1
    
    for name, details in menu_data.items():
        data_products.append({
            "product_id": product_id,
            "product_code": f"PCT{product_id:03d}",
            "product_name": name,
            "product_type": details.get("type"),
            "product_cat": details.get("cat"),
            "product_segment": details.get("price_bin"),
            "product_price": details.get("base_price")
        })
        product_id += 1
        
    return pd.DataFrame(data_products)

df_dict["products"]=build_products(MENU_DATA)

# =============
# 5. PROMOTIONS
# =============

def build_promotions(CONFIG):
    promotions = CONFIG["commercial"]["promotions"]
    data_promos = []
    
    for i, (code, details) in enumerate(promotions.items(), start=1):
        data_promos.append({
            "promotion_id": i,
            "promotion_code": code,
            "promotion_name": details.get("name"),
            "promotion_discount_pct": details.get("discount")
        })
        
    return pd.DataFrame(data_promos)

df_dict["promotions"] = build_promotions(CONFIG)

# ==========
# 6. RECIPES
# ==========

def build_recipes(menu_data, df_products, df_ingredients):
    data_recipes = []
    
    prod_lookup = df_products.set_index("product_name")["product_id"].to_dict()
    ing_lookup = df_ingredients.set_index("ingredient_name")["ingredient_id"].to_dict()
    
    for i, (p_name, details) in enumerate(menu_data.items(), start=1):
        if p_name in prod_lookup:
            product_id = prod_lookup[p_name]
            
            for ing_name, ing_info in details.get("recipe", {}).items():
                if ing_name in ing_lookup:
                    data_recipes.append({
                        "recipe_id": i,
                        "recipe_code": f"RCP{i:03d}",
                        "product_id": product_id,
                        "ingredient_id": ing_lookup[ing_name],
                        "recipe_quantity": ing_info["qty"],
                        "recipe_unit": ing_info["unit"]
                    })
                    
    return pd.DataFrame(data_recipes)

df_dict["recipes"]=build_recipes(MENU_DATA, df_dict["products"], df_dict["ingredients"])

# ===========
# 7. SERVICES
# ===========

def build_services(CONFIG):
    services = CONFIG["activity"]["services"]
    data_services = []
    
    for i, (code, details) in enumerate(services.items(), start=1):
        data_services.append({
            "service_id": i,
            "service_code": code,
            "service_name": details.get("name"),
            "service_rotation": details.get("rotation"),
            "service_group": "Cocktails" if code == "SRV002" else "Repas"
        })

    return pd.DataFrame(data_services)

df_dict["services"] = build_services(CONFIG)

# =========
# 8. TABLES
# =========

def build_tables(CONFIG):
    table_id = 1
    data_tables = []
    config_tables = CONFIG["infrastructure"]["tables"]
    
    for zone, capacities in config_tables.items():
        for cap, quantity in capacities.items():
            for _ in range(quantity):
                data_tables.append({
                    "table_id": table_id,
                    "table_code": f"TBL{table_id:03d}",
                    "table_zone": zone,
                    "table_capacity": int(cap)
                })
                table_id += 1
                
    return pd.DataFrame(data_tables)

df_dict["tables"] = build_tables(CONFIG)


# ETAPE 4 : MOTEUR DE SIMULATION DES COMMANDES, STOCKS ET DES PAYMENTS

In [ ]:
# ========================================
# 0. PREPARATION DES DONNEES DE SIMULATION
# ========================================

REF = {
    "products": df_dict["products"].set_index("product_id")[["product_price", "product_cat", "product_type"]].to_dict(orient="index"),
}

recipes_dict = (
    df_dict["recipes"]
    .groupby("product_id")
    .apply(lambda x: x[['ingredient_id', 'recipe_quantity']].values.tolist())
    .to_dict()
)

products_starter = df_dict["products"].loc[df_dict["products"]["product_cat"].isin(["Entrée"]), "product_id"].tolist()
products_dish = df_dict["products"].loc[df_dict["products"]["product_cat"].isin(["Plat"]), "product_id"].tolist()
products_dessert = df_dict["products"].loc[df_dict["products"]["product_cat"] == "Dessert", "product_id"].tolist()
products_beverage = df_dict["products"].loc[df_dict["products"]["product_cat"] == "Boisson", "product_id"].tolist()

# =============================================================
# 1. SIMULATION DES OPERATIONS (COMMANDES + PAIEMENTS + STOCKS)
# =============================================================

def build_operations(CONFIG, REF, recipes_dict, products_starter, products_dish, dessert_ids, beverage_ids):
    # =====================================================
    # 1. PREPARATION DES REFERENTIELS
    # =====================================================
    
    # Calendrier
    calendar = CONFIG["calendar"]
    dates = pd.date_range(start=calendar["start_date"], end=calendar["end_date"])
    covid = calendar["exceptional_closure_periods"]["covid"]
    covid_closure_periods = [(pd.Timestamp(p["start"]), pd.Timestamp(p["end"])) for p in covid["closure_periods"]]
    valid_dates = [d for d in dates if d.dayofweek not in calendar["closed_days"] and not any(start <= d <= end for start, end in covid_closure_periods)]
    winter_months = calendar["winter_months"]

    # Activité
    activity = CONFIG["activity"]
    services = activity["services"]
    n_customers = activity["n_customers"]
    service_map = {code: idx + 1 for idx, code in enumerate(activity["services"].keys())}
    activity_coefficients = activity["activity_coefficients"]
    
    # Infrastructure
    infrastructure = CONFIG["infrastructure"]
    tables = infrastructure["tables"]
    employees = infrastructure["employees"]

    # Capacité
    capacity_by_zone = {
        zone: sum(int(size) * count for size, count in capacities.items())
        for zone, capacities in tables.items()
    }
    room_capacity = capacity_by_zone.get("Salle", 0)
    vip_area_capacity = capacity_by_zone.get("VIP", 0)
    terrace_capacity = capacity_by_zone.get("Terrasse", 0)

    df_tables = df_dict["tables"]
    winter_tables = df_tables[df_tables["table_zone"] != "Terrasse"]["table_id"].tolist()
    tables = df_tables["table_id"].tolist()

    # Employées
    n_teams = employees["n_teams"]
    role_weights = employees.get("role_weights", {"Serveur": 1.0, "Chef de rang": 1.0, "Responsable salle": 1.0})

    df_employees = df_dict["employees"]
    emp_roles = dict(zip(df_employees["employee_id"], df_employees["employee_role"]))

    df_salle = df_employees[df_employees["employee_department"] == "Salle"]
    salle_teams_ids = {
    team: df_salle[df_salle["employee_team"] == team]["employee_id"].tolist()
    for team in range(1, n_teams + 1)
    }
    all_salle_ids = df_salle["employee_id"].tolist()

    # Commercial
    commercial = CONFIG["commercial"]
    promotions = commercial["promotions"]
    
    promo_ids = [int(k.replace("PRM", "")) for k in promotions.keys()]
    promo_probs = [p["prob"] for p in promotions.values()]

    product_price = {pid: data["product_price"] for pid, data in REF["products"].items()}
    
    behavior = commercial["behavior"]

    # Finance
    financials = CONFIG["financials"]
    stock = financials.get("stock_management")

    loss_prob = stock["loss_probability"]
    loss_rate_min, loss_rate_max = stock["loss_rate_range"][0], stock["loss_rate_range"][1]
    volatility_factor = stock["volatility_factor"]
    
    # Transactions
    payment_methods = financials["payment_methods"]
    methods = list(payment_methods.keys())
    probs = list(payment_methods.values())

    # Initialisation
    sales, data_movements, order_total = [], {}, {}
    order_id = 0

    # =====================================================
    # 2. FONCTIONS INTERNES
    # =====================================================
   
    def choose_employee(hour):
        # Logique de sélection basée sur l'heure
        h = int(hour)
        available_employees = []

        for team_id in range(1, n_teams + 1):
            schedule = employees["schedules"]["Salle"][team_id]
            if schedule["start"] <= h < schedule["end"]:
                available_employees.extend(salle_teams_ids.get(team_id, []))

        if not available_employees:
            available_employees = all_salle_ids

        # Choix pondéré par les rôles
        weights = [role_weights.get(emp_roles[eid], 1.0) for eid in available_employees]
        probs = (weights / sum_weights) if (sum_weights := np.sum(weights)) > 0 else None
        
        return np.random.choice(available_employees, p=probs)

    def generate_products(service, pax):
        products = []
        if service != "SRV002":
            for _ in range(pax):
                products.append(np.random.choice(products_dish))
                if np.random.rand() < 0.5:
                    products.append(np.random.choice(products_starter))
                    
        if service == "SRV002":
            products.append(np.random.choice(beverage_ids))
        
        if np.random.rand() < behavior["probabilities"]["dessert"]: products.append(np.random.choice(dessert_ids))
        if np.random.rand() < behavior["probabilities"]["drink"]: products.append(np.random.choice(beverage_ids))
        
        return products

    # =====================================================
    # 3. GENERATION DES COMMANDES
    # =====================================================
    for date in valid_dates:
    
        capacity = room_capacity + vip_area_capacity + (0 if date.month in winter_months else terrace_capacity)
        covid_reduction_rate = (1 - covid["reduction_rate"]) if date.year in covid["years"] else 1
        day_coeff = activity_coefficients.get(date.dayofweek, 1.0)
        daily_capacity = capacity * covid_reduction_rate * day_coeff

        for service_code, info in services.items():
            duration = info["end"] - info["start"]
            average_table_turn_time = max(1, int(duration / info["rotation"]))
            
            for start_hour in range(info["start"], info["end"], average_table_turn_time):
                min_rate, max_rate = info["occupancy_rate"]
                service_occupancity_rate = np.random.uniform(min_rate, max_rate)
                covers_per_turn = max(1, int(daily_capacity * service_occupancity_rate))
                current_covers_service = 0

                while current_covers_service < covers_per_turn:
                    pax = np.random.choice(behavior["pax"]["sizes"], p=behavior["pax"]["probs"])
                
                    if current_covers_service + pax > covers_per_turn:
                        break
            
                    current_covers_service += pax

                    # # Heure et date de la commande
                    order_hour = np.random.randint(start_hour, min(start_hour + average_table_turn_time, info["end"]))
                    order_datetime = date.replace(hour=order_hour, minute=np.random.randint(0, 60), second=0)
                    order_date = date.replace(hour=0, minute=0, second=0, microsecond=0)
                    dt_hour = date.replace(hour=order_hour, minute=0, second=0)

                    # Attribution d'une table pour toute la commande
                    table_id = np.random.choice(winter_tables if date.month in winter_months else tables)

                    # Attribution d'un client pour toute la commande
                    customer_id = np.random.randint(1, n_customers + 1)

                    # Attributuion d'un employé
                    employee_id = choose_employee(order_hour)
                    if employee_id is None:
                         error_msg = f"Impossible d'assigner un employé le {date.date()} à {order_hour}h pour le service {service_code}."
                         raise ValueError(error_msg)

                    # Génération commande
                    promo_id = np.random.choice(promo_ids, p=promo_probs)
                    products = generate_products(service_code, pax)
                    if not products:
                        raise ValueError(f"Échec de génération de commande le {date.date()} à {order_hour}h pour {pax} pax (Service {service_code}).")

                    order_id += 1
                    discount = promotions[f"PRM{promo_id:03d}"]["discount"]
                    order_total[order_id] = 0

                    for product_id in products:
                        price = product_price[product_id]
                        net_price = round(price * (1 - discount), 2)
                        order_total[order_id] += net_price

                        sales.append({
                        "order_id": order_id,
                        "order_code": f"ORD{order_id:05d}",
                        "datetime": order_datetime,
                        "date": order_date,
                        "hour": order_hour,
                        "product_id": product_id,
                        "customer_id": customer_id,
                        "table_id": table_id,
                        "service_id": service_map[service_code],
                        "promotion_id": promo_id,
                        "employee_id": employee_id,
                        "product_price_brut": price,
                        "product_price_net": net_price,
                        "product_quantity": 1,
                        "order_pax": pax
                        })

                        for ing_id, qty in recipes_dict.get(product_id, []):
                            key = (dt_hour, ing_id, "Consommation")
                            data_movements[key] = data_movements.get(key, 0) - qty

                            if np.random.rand() < loss_prob:
                                stress_factor = (current_covers_service / covers_per_turn)
                                dynamic_loss_rate = np.random.uniform(loss_rate_min, loss_rate_max) * (1 + (stress_factor * volatility_factor))
                                key_perte = (dt_hour, ing_id, "Perte")
                                data_movements[key_perte] = data_movements.get(key_perte, 0) - (qty * dynamic_loss_rate)

                    # Réapprovisionnement simulé au début du service
                    if order_hour == info["start"]:
                        for ing_id in range(1, 10): # Simule un réappro sur les 10 premiers ingrédients
                            k_rep = (date.replace(hour=info["start"], minute=0, second=0), ing_id, "Réapprovisionnement")
                            data_movements[k_rep] = data_movements.get(k_rep, 0) + 100
            
    stock_movements = [{
        "stock_movements_id" :i + 1,
        "stock_movements_code": f"AGG{i + 1:07d}",
        "datetime": k[0],
        "date": k[0].replace(hour=0, minute=0, second=0, microsecond=0),
        "hour": k[0].hour,
        "ingredient_id": k[1],
        "stock_movements_type": k[2],
        "stock_movements_quantity": v}
    for i, (k, v) in enumerate(data_movements.items())
    ]

    print(data_movements)

    return pd.DataFrame(sales), \
           pd.DataFrame(stock_movements)

df_dict["sales"], df_dict["stock_movements"] = build_operations(CONFIG, REF, recipes_dict, products_starter, products_dish, products_dessert, products_beverage)

# ETAPE 5 : EXPORT DES DONNEES

In [ ]:
# ================================
# 0. INITIALISATION DE LA FONCTION
# ================================

def export_data(df_dict):
    output_path = "Data/Raw"
    os.makedirs(output_path, exist_ok=True)
    
    print(f"⏳ Dataset en cours d'exportation dans : {output_path}\n")

    for name, df in df_dict.items():
        file_path = os.path.join(output_path, f"{name}.csv")
        df.to_csv(file_path, index=False)
        print(f"📄 Exporté : {name}.csv ({len(df)} lignes)")
    
    print(f"\n✅ Dataset complet exporté dans : {output_path}")

# =========================
# 1. EXPORT DE CHAQUE TABLE
# =========================

export_data(df_dict)